# Process Mining on Bitcoin Agent Event Log with PM4Py

This notebook analyzes the Bitcoin mining-pool agent event log from five angles:

1. Discover the agent decision process.
2. Check conformance against the intended behavior model.
3. Compare variants by agent type.
4. Analyze the decision point: skip vs mine.
5. Link process outcomes to profitability and market conditions.

## Important interpretation note

The case notion used here is:

> one agent / mining-pool decision for one block opportunity

So a case is not one physical production order. It is one simulated agent-block decision process.

Also, the events inside one case often share the same timestamp. Therefore, this notebook avoids activity-duration/performance mining and focuses on:
- control-flow variants,
- conformance logic,
- agent archetype comparison,
- decision mining,
- outcome/profit analysis,
- market-condition analysis.

## 0. Setup

Run the install cell once if PM4Py is not installed in your environment.

For a local environment, recommended Python version is 3.10 or newer.

In [ ]:
# Uncomment and run once if needed:
# %pip install pm4py pandas numpy matplotlib scikit-learn

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pm4py
    PM4PY_AVAILABLE = True
    print("PM4Py version:", getattr(pm4py, "__version__", "unknown"))
except ModuleNotFoundError:
    PM4PY_AVAILABLE = False
    print("PM4Py is not installed. Run the install cell above and restart the notebook kernel.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

OUTPUT_DIR = Path("pm4py_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# The notebook first looks for the dataset in the same folder as the notebook.
# In ChatGPT's sandbox it also works with /mnt/data/.
DATA_PATH = Path("bitcoin_agent_event_log_historical_playthrough_archetypes.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/MAS/bitcoin_agent_event_log_historical_playthrough_archetypes.csv")

DATA_PATH

## 1. Load and inspect the event log

Required process-mining columns in this dataset:

| Dataset column | Process-mining meaning |
|---|---|
| `case_id` | Case identifier: one agent-block decision |
| `activity` | Activity name |
| `timestamp` | Event timestamp |
| `event_id` | Ordering column inside and across cases |
| `agent_name` | Resource / mining-pool agent |
| `agent_type` | Organizational group / archetype |

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Rows / events:", len(df))
print("Columns:", len(df.columns))
display(df.head())
display(df.dtypes.to_frame("dtype"))

In [ ]:
# Basic data quality checks
required_cols = ["case_id", "activity", "timestamp", "event_id", "agent_name", "agent_type"]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

overview = pd.DataFrame({
    "metric": [
        "events",
        "cases",
        "activities",
        "agents",
        "agent types",
        "blocks / steps",
        "timestamp min",
        "timestamp max",
        "missing timestamps"
    ],
    "value": [
        len(df),
        df["case_id"].nunique(),
        df["activity"].nunique(),
        df["agent_name"].nunique(),
        df["agent_type"].nunique(),
        df["step"].nunique() if "step" in df.columns else np.nan,
        df["timestamp"].min(),
        df["timestamp"].max(),
        df["timestamp"].isna().sum()
    ]
})
display(overview)
display(df["activity"].value_counts().rename_axis("activity").reset_index(name="events"))
display(df["agent_type"].value_counts().rename_axis("agent_type").reset_index(name="events"))

## 2. Prepare PM4Py-compatible event log

PM4Py expects three standard columns:

- `case:concept:name`
- `concept:name`
- `time:timestamp`

The dataset also contains `agent_name`, which we map to `org:resource`.

In [ ]:
event_log_df = df.copy()

event_log_df = event_log_df.rename(columns={
    "case_id": "case:concept:name",
    "activity": "concept:name",
    "timestamp": "time:timestamp",
    "agent_name": "org:resource"
})

# Sorting is important because activities in a case often have identical timestamps.
# event_id is therefore used as the primary within-case ordering.
event_log_df = event_log_df.sort_values(
    ["case:concept:name", "event_id"],
    kind="mergesort"
).reset_index(drop=True)

pm_cols = ["case:concept:name", "concept:name", "time:timestamp", "org:resource"]
display(event_log_df[pm_cols + ["event_id", "agent_type"]].head(10))

if PM4PY_AVAILABLE:
    # PM4Py can work directly with formatted pandas dataframes.
    # Sorting has already been done above.
    log = pm4py.convert_to_event_log(event_log_df)
    print("Converted to PM4Py EventLog with traces:", len(log))
else:
    log = None

## 3. Create a case-level table

Many analyses are easier at the case level.  
Each row below represents one agent-block decision process.

In [ ]:
# Trace extraction
trace_table = (
    event_log_df
    .groupby("case:concept:name")["concept:name"]
    .apply(tuple)
    .rename("trace")
    .reset_index()
)

trace_table["trace_string"] = trace_table["trace"].apply(lambda x: " → ".join(x))
trace_table["trace_length"] = trace_table["trace"].apply(len)

# Case-level static/context attributes:
# Use first non-null value where possible. For financial outcome columns, the last event is often most meaningful.
first_cols = [
    "agent_type", "org:resource", "step", "block_height",
    "btc_price_at_block", "transaction_fees_btc", "total_reward_btc",
    "block_interval_minutes", "network_hashrate_th_s",
    "selected_allocation", "potential_hashrate_th_s", "allocated_hashrate_th_s",
    "power_kw", "expected_win_probability_vs_historical_network",
    "expected_revenue_usd", "expected_cost_usd", "expected_profit_usd",
    "best_allocation_profit_usd"
]
first_cols = [c for c in first_cols if c in event_log_df.columns]

last_cols = [
    "realized_cost_usd", "realized_revenue_usd", "cumulative_profit_usd", "reward_btc"
]
last_cols = [c for c in last_cols if c in event_log_df.columns]

case_first = (
    event_log_df
    .groupby("case:concept:name")[first_cols]
    .first()
    .reset_index()
)

case_last = (
    event_log_df
    .sort_values(["case:concept:name", "event_id"])
    .groupby("case:concept:name")[last_cols]
    .last()
    .reset_index()
)

case_summary = (
    trace_table
    .merge(case_first, on="case:concept:name", how="left")
    .merge(case_last, on="case:concept:name", how="left")
)

def classify_outcome(trace):
    s = set(trace)
    if "Skip_Mining" in s:
        return "Skip"
    if "Win_Block" in s:
        return "Win"
    if "Lose_Block" in s:
        return "Lose"
    return "Other"

case_summary["outcome"] = case_summary["trace"].apply(classify_outcome)
case_summary["decision"] = np.where(case_summary["outcome"].eq("Skip"), "Skip", "Mine")

display(case_summary.head())
display(case_summary["outcome"].value_counts().rename_axis("outcome").reset_index(name="cases"))
display(case_summary["trace_string"].value_counts().rename_axis("variant").reset_index(name="cases"))

# Part 1 — Discover the agent decision process

Because there are only a few trace variants, the Directly-Follows Graph (DFG) is usually the clearest visualization.

In [ ]:
if PM4PY_AVAILABLE:
    dfg, start_activities, end_activities = pm4py.discover_dfg(log)

    print("Start activities:", start_activities)
    print("End activities:", end_activities)
    print("Directly-follows edges:", len(dfg))

    # Display in notebook if graphviz is available.
    try:
        pm4py.view_dfg(dfg, start_activities, end_activities)
    except Exception as e:
        print("Could not display DFG inline:", e)

    # Save visualization to file.
    try:
        dfg_path = OUTPUT_DIR / "01_dfg_agent_decision_process.png"
        pm4py.save_vis_dfg(dfg, start_activities, end_activities, str(dfg_path))
        print("Saved DFG to:", dfg_path)
    except Exception as e:
        print("Could not save DFG:", e)
else:
    print("Install PM4Py to run process discovery.")

In [ ]:
# Variant overview without relying on PM4Py
variant_counts = (
    case_summary["trace_string"]
    .value_counts()
    .rename_axis("variant")
    .reset_index(name="cases")
)
variant_counts["share"] = variant_counts["cases"] / variant_counts["cases"].sum()
display(variant_counts)

ax = variant_counts.sort_values("cases").plot(
    kind="barh",
    x="variant",
    y="cases",
    legend=False,
    figsize=(10, 4),
    title="Trace variants in the agent decision process"
)
ax.set_xlabel("Number of cases")
ax.set_ylabel("Variant")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: discover a Petri net using the Inductive Miner.
# This is mostly useful for conformance checking and for showing a formal process model.
if PM4PY_AVAILABLE:
    try:
        net, im, fm = pm4py.discover_petri_net_inductive(log)
        print("Discovered Petri net with Inductive Miner.")
        try:
            pm4py.view_petri_net(net, im, fm)
        except Exception as e:
            print("Could not display Petri net inline:", e)
        try:
            petri_path = OUTPUT_DIR / "02_inductive_miner_petri_net.png"
            pm4py.save_vis_petri_net(net, im, fm, str(petri_path))
            print("Saved Petri net to:", petri_path)
        except Exception as e:
            print("Could not save Petri net:", e)
    except Exception as e:
        print("Could not discover Petri net:", e)

# Part 2 — Conformance checking against the intended behavior model

The intended process contains exactly three valid patterns:

1. Skip path  
   `Observe_Market → Evaluate_Hashrate_Allocation → Choose_Hashrate_Allocation → Skip_Mining`

2. Mine and lose path  
   `Observe_Market → Evaluate_Hashrate_Allocation → Choose_Hashrate_Allocation → Start_Mining → Consume_Electricity → Lose_Block → Update_Profit`

3. Mine and win path  
   `Observe_Market → Evaluate_Hashrate_Allocation → Choose_Hashrate_Allocation → Start_Mining → Consume_Electricity → Win_Block → Receive_Reward → Update_Profit`

This section performs two types of conformance checking:

- variant-level conformance: does the trace exactly match one expected trace?
- rule-level conformance: does the log violate important behavioral rules?

In [ ]:
EXPECTED_TRACES = {
    (
        "Observe_Market",
        "Evaluate_Hashrate_Allocation",
        "Choose_Hashrate_Allocation",
        "Skip_Mining"
    ): "Expected skip path",
    (
        "Observe_Market",
        "Evaluate_Hashrate_Allocation",
        "Choose_Hashrate_Allocation",
        "Start_Mining",
        "Consume_Electricity",
        "Lose_Block",
        "Update_Profit"
    ): "Expected mine-loss path",
    (
        "Observe_Market",
        "Evaluate_Hashrate_Allocation",
        "Choose_Hashrate_Allocation",
        "Start_Mining",
        "Consume_Electricity",
        "Win_Block",
        "Receive_Reward",
        "Update_Profit"
    ): "Expected mine-win path"
}

case_summary["expected_trace"] = case_summary["trace"].isin(EXPECTED_TRACES.keys())
case_summary["expected_trace_label"] = case_summary["trace"].map(EXPECTED_TRACES).fillna("Unexpected trace")

conformance_summary = (
    case_summary["expected_trace_label"]
    .value_counts()
    .rename_axis("trace_conformance")
    .reset_index(name="cases")
)
conformance_summary["share"] = conformance_summary["cases"] / conformance_summary["cases"].sum()
display(conformance_summary)

unexpected_cases = case_summary.loc[~case_summary["expected_trace"]]
print("Unexpected cases:", len(unexpected_cases))
display(unexpected_cases.head(20))

In [ ]:
# Rule-level conformance checks
case_activity_sets = (
    event_log_df
    .groupby("case:concept:name")["concept:name"]
    .apply(set)
    .rename("activity_set")
    .reset_index()
)

case_order = (
    event_log_df
    .groupby("case:concept:name")["concept:name"]
    .apply(list)
    .rename("activity_list")
    .reset_index()
)

rules = case_summary[["case:concept:name", "trace", "outcome", "decision", "agent_type"]].copy()
rules = rules.merge(case_activity_sets, on="case:concept:name", how="left")
rules = rules.merge(case_order, on="case:concept:name", how="left")

def has(row, activity):
    return activity in row["activity_set"]

rules["starts_with_observe"] = rules["trace"].apply(lambda t: len(t) > 0 and t[0] == "Observe_Market")
rules["ends_validly"] = rules["trace"].apply(lambda t: len(t) > 0 and t[-1] in {"Skip_Mining", "Update_Profit"})
rules["skip_did_not_consume"] = ~rules.apply(lambda r: has(r, "Skip_Mining") and has(r, "Consume_Electricity"), axis=1)
rules["win_has_reward"] = rules.apply(lambda r: (not has(r, "Win_Block")) or has(r, "Receive_Reward"), axis=1)
rules["reward_requires_win"] = rules.apply(lambda r: (not has(r, "Receive_Reward")) or has(r, "Win_Block"), axis=1)
rules["lose_has_no_reward"] = rules.apply(lambda r: not (has(r, "Lose_Block") and has(r, "Receive_Reward")), axis=1)
rules["mining_requires_consumption"] = rules.apply(lambda r: (not has(r, "Start_Mining")) or has(r, "Consume_Electricity"), axis=1)

rule_cols = [
    "starts_with_observe",
    "ends_validly",
    "skip_did_not_consume",
    "win_has_reward",
    "reward_requires_win",
    "lose_has_no_reward",
    "mining_requires_consumption"
]

rule_summary = pd.DataFrame({
    "rule": rule_cols,
    "fitting_cases": [rules[c].sum() for c in rule_cols],
    "violating_cases": [(~rules[c]).sum() for c in rule_cols],
})
rule_summary["fit_share"] = rule_summary["fitting_cases"] / len(rules)
display(rule_summary)

violations = rules.loc[~rules[rule_cols].all(axis=1), ["case:concept:name", "agent_type", "outcome", "trace"] + rule_cols]
print("Cases violating at least one rule:", len(violations))
display(violations.head(20))

In [ ]:
# Block-level conformance: exactly one winner per block
if "block_height" in event_log_df.columns:
    win_events = event_log_df[event_log_df["concept:name"].eq("Win_Block")].copy()
    winners_per_block = (
        win_events
        .groupby("block_height")
        .agg(
            winners=("case:concept:name", "nunique"),
            winning_agents=("org:resource", lambda x: list(x.unique())),
            winning_agent_types=("agent_type", lambda x: list(x.unique()))
        )
        .reset_index()
    )

    all_blocks = pd.DataFrame({"block_height": sorted(event_log_df["block_height"].dropna().unique())})
    winners_per_block = all_blocks.merge(winners_per_block, on="block_height", how="left")
    winners_per_block["winners"] = winners_per_block["winners"].fillna(0).astype(int)

    block_conf = winners_per_block["winners"].value_counts().sort_index().rename_axis("winners_per_block").reset_index(name="blocks")
    display(block_conf)

    bad_blocks = winners_per_block[winners_per_block["winners"] != 1]
    print("Blocks with not exactly one winner:", len(bad_blocks))
    display(bad_blocks.head(20))

## Optional PM4Py token-based replay against an intended model

The rule checks above are faster and easier to explain.  
If you also want a formal PM4Py conformance result, the code below builds a small expected log with the three allowed variants, discovers a Petri net from that expected log, and replays a sample of actual cases against it.

Set `RUN_FULL_TOKEN_REPLAY = True` only if you want to run it on the full log. For large logs this can take time.

In [ ]:
RUN_FULL_TOKEN_REPLAY = False
TOKEN_REPLAY_SAMPLE_CASES = 5000

if PM4PY_AVAILABLE:
    try:
        from pm4py.objects.log.obj import EventLog, Trace, Event

        expected_log = EventLog()
        for trace_tuple in EXPECTED_TRACES.keys():
            tr = Trace()
            for act in trace_tuple:
                tr.append(Event({"concept:name": act, "time:timestamp": pd.Timestamp("2025-01-01", tz="UTC")}))
            expected_log.append(tr)

        expected_net, expected_im, expected_fm = pm4py.discover_petri_net_inductive(expected_log)

        if RUN_FULL_TOKEN_REPLAY:
            replay_log = log
            print("Running token replay on full log.")
        else:
            sample_cases = case_summary["case:concept:name"].sample(
                min(TOKEN_REPLAY_SAMPLE_CASES, len(case_summary)),
                random_state=42
            )
            replay_df = event_log_df[event_log_df["case:concept:name"].isin(sample_cases)].copy()
            replay_log = pm4py.convert_to_event_log(replay_df)
            print(f"Running token replay on sample of {len(sample_cases)} cases.")

        fitness = pm4py.fitness_token_based_replay(replay_log, expected_net, expected_im, expected_fm)
        display(pd.DataFrame([fitness]))

    except Exception as e:
        print("Token-based replay could not be completed:", e)

# Part 3 — Compare variants by agent type

This section compares how often each archetype follows the skip, mine-loss, or mine-win path.

In [ ]:
variant_by_type = pd.crosstab(
    case_summary["agent_type"],
    case_summary["outcome"],
    margins=True
)

variant_by_type_pct = pd.crosstab(
    case_summary["agent_type"],
    case_summary["outcome"],
    normalize="index"
)

display(variant_by_type)
display((variant_by_type_pct * 100).round(2))

# Plot outcome shares by agent type
plot_df = variant_by_type_pct.drop(index="All", errors="ignore") if "All" in variant_by_type_pct.index else variant_by_type_pct
ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 5),
    title="Outcome share by agent type"
)
ax.set_ylabel("Share of cases")
ax.set_xlabel("Agent type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# More detailed trace variant comparison by agent type
variant_type_table = (
    case_summary
    .groupby(["agent_type", "trace_string"])
    .size()
    .reset_index(name="cases")
)

variant_type_table["share_within_agent_type"] = (
    variant_type_table["cases"] /
    variant_type_table.groupby("agent_type")["cases"].transform("sum")
)

display(
    variant_type_table
    .sort_values(["agent_type", "cases"], ascending=[True, False])
)

# Agent-level outcome performance
agent_perf = (
    case_summary
    .groupby(["agent_type", "org:resource", "outcome"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["Skip", "Lose", "Win"]:
    if col not in agent_perf.columns:
        agent_perf[col] = 0

agent_perf["cases"] = agent_perf[["Skip", "Lose", "Win"]].sum(axis=1)
agent_perf["skip_rate"] = agent_perf["Skip"] / agent_perf["cases"]
agent_perf["win_rate"] = agent_perf["Win"] / agent_perf["cases"]
agent_perf["mine_rate"] = 1 - agent_perf["skip_rate"]

display(agent_perf.sort_values(["agent_type", "win_rate"], ascending=[True, False]).head(30))

# Part 4 — Analyze the decision point: skip vs mine

The core decision point is:

`Choose_Hashrate_Allocation → Skip_Mining`  
or  
`Choose_Hashrate_Allocation → Start_Mining`

This section analyzes which variables separate skip cases from mining cases.

In [ ]:
decision_features = [
    "selected_allocation",
    "allocated_hashrate_th_s",
    "power_kw",
    "expected_win_probability_vs_historical_network",
    "expected_revenue_usd",
    "expected_cost_usd",
    "expected_profit_usd",
    "best_allocation_profit_usd",
    "btc_price_at_block",
    "transaction_fees_btc",
    "total_reward_btc",
    "network_hashrate_th_s",
    "block_interval_minutes"
]
decision_features = [c for c in decision_features if c in case_summary.columns]

decision_df = case_summary[
    ["case:concept:name", "agent_type", "org:resource", "decision", "outcome"] + decision_features
].copy()

display(decision_df.head())

# Main decision summary
decision_summary = (
    decision_df
    .groupby(["agent_type", "decision"])
    .size()
    .reset_index(name="cases")
)
decision_summary["share_within_agent_type"] = (
    decision_summary["cases"] /
    decision_summary.groupby("agent_type")["cases"].transform("sum")
)
display(decision_summary)

# Numeric differences between skip and mine
numeric_decision_summary = (
    decision_df
    .groupby("decision")[decision_features]
    .agg(["mean", "median", "std"])
    .T
)
display(numeric_decision_summary)

In [ ]:
# Visualize the strongest direct variable: selected allocation
if "selected_allocation" in decision_df.columns:
    ax = decision_df.boxplot(
        column="selected_allocation",
        by="decision",
        figsize=(7, 5)
    )
    ax.set_title("Selected allocation by decision")
    ax.set_xlabel("Decision")
    ax.set_ylabel("Selected allocation")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

# Expected profit by decision
if "expected_profit_usd" in decision_df.columns:
    ax = decision_df.boxplot(
        column="expected_profit_usd",
        by="decision",
        figsize=(7, 5)
    )
    ax.set_title("Expected profit by decision")
    ax.set_xlabel("Decision")
    ax.set_ylabel("Expected profit USD")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

## Optional: simple decision model

This is not process discovery. It is decision mining / predictive explanation.

The goal is to test whether market and agent variables explain the branch:

- `Skip`
- `Mine`

Because the decision rules may be baked into the simulation, very high accuracy should not be interpreted as a real-world causal discovery. It means the generated behavior is strongly recoverable from the logged variables.

In [ ]:
RUN_DECISION_MODEL = True
MAX_MODEL_ROWS = 60000

if RUN_DECISION_MODEL:
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.compose import ColumnTransformer
        from sklearn.preprocessing import OneHotEncoder, StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

        model_df = decision_df.dropna(subset=["decision"]).copy()
        model_df["target_mine"] = (model_df["decision"] == "Mine").astype(int)

        feature_cols = decision_features + ["agent_type"]
        model_df = model_df.dropna(subset=feature_cols)

        if len(model_df) > MAX_MODEL_ROWS:
            model_df = model_df.sample(MAX_MODEL_ROWS, random_state=42)

        X = model_df[feature_cols]
        y = model_df["target_mine"]

        numeric_features = [c for c in decision_features if c in X.columns]
        categorical_features = ["agent_type"]

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), numeric_features),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
            ]
        )

        clf = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        print("Classification report: target 1 = Mine, target 0 = Skip")
        print(classification_report(y_test, y_pred))
        print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 4))
        print("Confusion matrix:")
        display(pd.DataFrame(
            confusion_matrix(y_test, y_pred),
            index=["actual_skip", "actual_mine"],
            columns=["pred_skip", "pred_mine"]
        ))

        # Coefficient interpretation
        feature_names = (
            numeric_features +
            list(clf.named_steps["preprocess"]
                 .named_transformers_["cat"]
                 .get_feature_names_out(categorical_features))
        )
        coefs = clf.named_steps["model"].coef_[0]
        coef_df = pd.DataFrame({
            "feature": feature_names,
            "coefficient_for_mine": coefs,
            "abs_coefficient": np.abs(coefs)
        }).sort_values("abs_coefficient", ascending=False)

        display(coef_df.head(25))

    except Exception as e:
        print("Decision model could not be completed:", e)

# Part 5 — Link process outcomes to profitability and market conditions

This section treats the process path as an outcome:

- `Skip`
- `Lose`
- `Win`

Then it compares process outcomes on financial and market variables.

In [ ]:
outcome_features = [
    "selected_allocation",
    "allocated_hashrate_th_s",
    "expected_win_probability_vs_historical_network",
    "expected_revenue_usd",
    "expected_cost_usd",
    "expected_profit_usd",
    "best_allocation_profit_usd",
    "realized_cost_usd",
    "realized_revenue_usd",
    "reward_btc",
    "cumulative_profit_usd",
    "btc_price_at_block",
    "transaction_fees_btc",
    "total_reward_btc",
    "network_hashrate_th_s",
    "block_interval_minutes"
]
outcome_features = [c for c in outcome_features if c in case_summary.columns]

outcome_summary = (
    case_summary
    .groupby("outcome")[outcome_features]
    .agg(["count", "mean", "median", "std"])
)

display(outcome_summary)

In [ ]:
# Outcome summary by agent type
outcome_by_type_financial = (
    case_summary
    .groupby(["agent_type", "outcome"])[outcome_features]
    .mean()
    .reset_index()
)

display(outcome_by_type_financial)

In [ ]:
# Plot expected profit by outcome
if "expected_profit_usd" in case_summary.columns:
    ax = case_summary.boxplot(
        column="expected_profit_usd",
        by="outcome",
        figsize=(8, 5)
    )
    ax.set_title("Expected profit by process outcome")
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Expected profit USD")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

# Plot realized cost by outcome
if "realized_cost_usd" in case_summary.columns:
    ax = case_summary.boxplot(
        column="realized_cost_usd",
        by="outcome",
        figsize=(8, 5)
    )
    ax.set_title("Realized cost by process outcome")
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Realized cost USD")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

# Plot realized revenue by outcome
if "realized_revenue_usd" in case_summary.columns:
    ax = case_summary.boxplot(
        column="realized_revenue_usd",
        by="outcome",
        figsize=(8, 5)
    )
    ax.set_title("Realized revenue by process outcome")
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Realized revenue USD")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

## Market-condition bins

Here we bin market variables into quartiles and examine whether decision/outcome shares change across market regimes.

In [ ]:
market_vars = [
    "btc_price_at_block",
    "transaction_fees_btc",
    "total_reward_btc",
    "network_hashrate_th_s",
    "block_interval_minutes"
]
market_vars = [c for c in market_vars if c in case_summary.columns]

for var in market_vars:
    tmp = case_summary[["outcome", "decision", "agent_type", var]].dropna().copy()
    if tmp[var].nunique() < 4:
        print(f"Skipping {var}: fewer than 4 unique values.")
        continue

    tmp[f"{var}_quartile"] = pd.qcut(tmp[var], q=4, duplicates="drop")

    print("\n" + "="*90)
    print(f"Market regime analysis: {var}")

    decision_by_bin = pd.crosstab(
        tmp[f"{var}_quartile"],
        tmp["decision"],
        normalize="index"
    )
    display((decision_by_bin * 100).round(2))

    outcome_by_bin = pd.crosstab(
        tmp[f"{var}_quartile"],
        tmp["outcome"],
        normalize="index"
    )
    display((outcome_by_bin * 100).round(2))

    ax = decision_by_bin.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 4),
        title=f"Decision share by {var} quartile"
    )
    ax.set_ylabel("Share")
    ax.set_xlabel(f"{var} quartile")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

# Additional analysis — network/block-level checks

This section changes perspective from agent-case to block-level competition.

Use this to check:
- which archetype wins most blocks,
- whether higher-fee blocks attract more active mining,
- how many agents mine per block.

In [ ]:
block_level = (
    case_summary
    .groupby(["block_height"])
    .agg(
        block_time=("time:timestamp", "first") if "time:timestamp" in case_summary.columns else ("case:concept:name", "count"),
    )
) if False else None

# Build block-level table from case_summary
block_group_cols = ["block_height"]
if "block_height" in case_summary.columns:
    block_summary = (
        case_summary
        .groupby("block_height")
        .agg(
            cases=("case:concept:name", "nunique"),
            active_miners=("decision", lambda s: (s == "Mine").sum()),
            skipping_agents=("decision", lambda s: (s == "Skip").sum()),
            winners=("outcome", lambda s: (s == "Win").sum()),
            avg_expected_profit_usd=("expected_profit_usd", "mean") if "expected_profit_usd" in case_summary.columns else ("case:concept:name", "count"),
            btc_price_at_block=("btc_price_at_block", "first") if "btc_price_at_block" in case_summary.columns else ("case:concept:name", "count"),
            transaction_fees_btc=("transaction_fees_btc", "first") if "transaction_fees_btc" in case_summary.columns else ("case:concept:name", "count"),
            network_hashrate_th_s=("network_hashrate_th_s", "first") if "network_hashrate_th_s" in case_summary.columns else ("case:concept:name", "count")
        )
        .reset_index()
    )

    win_cases = case_summary[case_summary["outcome"].eq("Win")][["block_height", "org:resource", "agent_type"]].copy()
    win_cases = win_cases.rename(columns={
        "org:resource": "winning_agent",
        "agent_type": "winning_agent_type"
    })
    block_summary = block_summary.merge(win_cases, on="block_height", how="left")

    display(block_summary.head())
    display(block_summary["winning_agent_type"].value_counts(dropna=False).rename_axis("winning_agent_type").reset_index(name="blocks"))

    if "transaction_fees_btc" in block_summary.columns:
        ax = block_summary.plot(
            kind="scatter",
            x="transaction_fees_btc",
            y="active_miners",
            figsize=(8, 5),
            title="Active mining agents per block vs transaction fees"
        )
        ax.set_xlabel("Transaction fees BTC")
        ax.set_ylabel("Active mining agents")
        plt.tight_layout()
        plt.show()

# Export analysis tables

The following cell saves the most useful tables to CSV files in `pm4py_outputs/`.

In [ ]:
exports = {
    "case_summary.csv": case_summary,
    "variant_counts.csv": variant_counts,
    "conformance_summary.csv": conformance_summary,
    "rule_summary.csv": rule_summary,
    "variant_by_agent_type_counts.csv": variant_by_type.reset_index(),
    "variant_by_agent_type_percentages.csv": (variant_by_type_pct * 100).round(4).reset_index(),
    "decision_summary.csv": decision_summary,
    "outcome_by_type_financial_means.csv": outcome_by_type_financial
}

for filename, table in exports.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False)
    print("Saved:", path)

# Suggested interpretation for your report

You can frame the analysis as follows:

## Research angle

> Process mining is used to validate and analyze an agent-based simulation of Bitcoin mining-pool behavior under historical market conditions.

## Main findings to report

1. **The discovered process is simple but meaningful.**  
   The log contains three main variants: skip, mine-loss, and mine-win.

2. **Conformance is the key methodological contribution.**  
   Since the behavior was generated by agents, PM4Py is useful for validating whether the event log follows the intended behavioral protocol.

3. **Agent archetypes differ strongly.**  
   Dominant low-cost pools should mine more often and win more often; small outside-network pools should skip more often and win rarely.

4. **The decision point is the most important analytical location.**  
   The branch after `Choose_Hashrate_Allocation` can be analyzed using selected allocation, expected cost, expected revenue, expected profit, BTC price, transaction fees, and agent type.

5. **Outcome mining links process paths to economic meaning.**  
   Skip cases avoid direct mining cost, mine-loss cases incur cost without reward, and mine-win cases receive block reward and fees.

## Limitation

Do not claim that PM4Py discovers unknown real-world miner behavior here.  
The stronger claim is:

> PM4Py helps inspect, validate, and compare the simulated behavior generated by the mining-pool agents.